# Cross Validation

## Learning Objectives
- Understand why selecting a model by minimising training error always picks the most complex model
- Implement **hold-out (simple) cross validation** and understand the 70/30 train/CV split
- Implement **k-fold cross validation** and understand how it reduces the data waste of hold-out CV
- Understand **leave-one-out cross validation** as the extreme case $k = m$
- Use cross validation to select polynomial degree as a model selection problem

## Problem Statement

Given a finite set of models $\mathcal{M} = \{M_1, \ldots, M_d\}$ (e.g. polynomials of degree 0 through 10), select the model $M_i$ that best balances bias and variance.

**Why training-error selection fails:**
Higher-order polynomials always fit training data better, so $\arg\min_i \hat{\varepsilon}_{S}(h_i)$ always picks the most complex model — a guaranteed overfitter.

**The fix — evaluate on held-out data:**

| Method | Data split | Cost |
|---|---|---|
| Hold-out CV | Once: 70% train / 30% CV | Train each model once |
| $k$-fold CV | $k$ rotations; each fold is test once | Train each model $k$ times |
| Leave-one-out (LOOCV) | $k = m$; test on single example | Train each model $m$ times |

## 1. Why Training-Error Model Selection Fails

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)

def true_fn(x):
    return np.sin(2 * np.pi * x)

m = 15
X = np.sort(np.random.uniform(0, 1, m))
y = true_fn(X) + np.random.normal(0, 0.3, m)

degrees      = list(range(1, 13))
train_errors = []

for deg in degrees:
    p = np.poly1d(np.polyfit(X, y, deg))
    train_errors.append(np.mean((p(X) - y) ** 2))

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(degrees, train_errors, 'b-o', lw=2.5, ms=7)
ax.set_xlabel('Polynomial degree')
ax.set_ylabel('Training MSE')
ax.set_title('Training error always decreases with model complexity\n'
             '→ selecting by min train error always picks highest degree')
ax.annotate('Always picks this!',
            xy=(degrees[-1], train_errors[-1]),
            xytext=(degrees[-1] - 4, train_errors[-1] + 0.05),
            fontsize=10, color='red',
            arrowprops=dict(arrowstyle='->', color='red', lw=1.5))
ax.text(0.03, 0.97,
        'Training error is a biased estimator of generalisation error\n'
        '— we need held-out data to evaluate models fairly.',
        transform=ax.transAxes, fontsize=9, va='top',
        bbox=dict(boxstyle='round', fc='lightyellow', alpha=0.9))
plt.tight_layout()
plt.show()

## 2. Hold-Out Cross Validation

**Algorithm:**
1. Randomly split $S$ into $S_{\text{train}}$ (70%) and $S_{\text{cv}}$ (30%)
2. Train each model $M_i$ on $S_{\text{train}}$ only → hypothesis $h_i$
3. Select $h_i$ with smallest $\hat{\varepsilon}_{S_{\text{cv}}}(h_i)$
4. Optionally retrain selected $M_i$ on full $S$

**Drawback:** effectively uses only 70% of the data for training when the dataset is small.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

def true_fn(x):
    return np.sin(2 * np.pi * x)

m = 40
X_all = np.sort(np.random.uniform(0, 1, m))
y_all = true_fn(X_all) + np.random.normal(0, 0.3, m)

# Hold-out split: 70% train, 30% CV
split   = int(0.7 * m)
idx     = np.random.permutation(m)
tr_idx  = idx[:split]
cv_idx  = idx[split:]

X_tr, y_tr = X_all[tr_idx], y_all[tr_idx]
X_cv, y_cv = X_all[cv_idx], y_all[cv_idx]

degrees      = list(range(1, 12))
train_errors = []
cv_errors    = []

for deg in degrees:
    p = np.poly1d(np.polyfit(X_tr, y_tr, deg))
    train_errors.append(np.mean((p(X_tr) - y_tr) ** 2))
    cv_errors.append(np.mean((p(X_cv) - y_cv) ** 2))

best_deg = degrees[np.argmin(cv_errors)]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: error curves
ax = axes[0]
ax.plot(degrees, train_errors, 'b-o', lw=2, ms=6, label='Train error')
ax.plot(degrees, cv_errors,    'r-s', lw=2, ms=6, label='CV error')
ax.axvline(best_deg, color='green', ls='--', lw=2,
           label=f'Best degree = {best_deg}')
ax.set_xlabel('Polynomial degree'); ax.set_ylabel('MSE')
ax.set_title('Hold-Out Cross Validation\nTrain/CV error vs. polynomial degree')
ax.legend(fontsize=9)
ax.set_ylim(0, min(2.0, max(cv_errors) * 1.2))

# Right: best fit on full data
ax = axes[1]
x_plot = np.linspace(0, 1, 300)
p_best = np.poly1d(np.polyfit(X_tr, y_tr, best_deg))
ax.plot(x_plot, true_fn(x_plot), 'g-',  lw=2, label='True function')
ax.plot(x_plot, p_best(x_plot),  'b-',  lw=2.5, label=f'Degree-{best_deg} fit (selected)')
ax.scatter(X_tr, y_tr, c='blue', s=50, alpha=0.7, label='Train set', zorder=5)
ax.scatter(X_cv, y_cv, c='red',  s=70, marker='s', alpha=0.8, label='CV set', zorder=5)
ax.set_xlabel('$x$'); ax.set_ylabel('$y$')
ax.set_title(f'Selected model (degree {best_deg}) fitted on $S_{{\\rm train}}$\nRed squares: hold-out CV set')
ax.legend(fontsize=9); ax.set_ylim(-2.5, 2.5)

fig.suptitle('Hold-Out Cross Validation for Model Selection', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()
print(f'Selected polynomial degree: {best_deg}')

## 3. k-Fold Cross Validation

**Algorithm:**
1. Split $S$ into $k$ disjoint subsets $S_1, \ldots, S_k$ of size $m/k$ each
2. For each model $M_i$ and each fold $j = 1, \ldots, k$:
   - Train on $S \setminus S_j$, test on $S_j$ → error $\hat{\varepsilon}_{S_j}(h_{ij})$
3. Estimated generalisation error of $M_i$ = $\frac{1}{k}\sum_j \hat{\varepsilon}_{S_j}(h_{ij})$
4. Pick $M_i$ with lowest estimated error, retrain on all of $S$

Typical choice: $k = 10$. Extreme: $k = m$ (leave-one-out).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error

np.random.seed(7)

def true_fn(x):
    return np.sin(2 * np.pi * x)

m = 40
X_all = np.sort(np.random.uniform(0, 1, m)).reshape(-1, 1)
y_all = true_fn(X_all.ravel()) + np.random.normal(0, 0.3, m)

degrees  = list(range(1, 12))
k_values = [3, 5, 10]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: CV error for different k values
ax = axes[0]
for k in k_values:
    kf       = KFold(n_splits=k, shuffle=True, random_state=0)
    cv_errs  = []
    for deg in degrees:
        fold_errs = []
        for tr_idx, te_idx in kf.split(X_all):
            model = make_pipeline(PolynomialFeatures(deg), LinearRegression())
            model.fit(X_all[tr_idx], y_all[tr_idx])
            fold_errs.append(mean_squared_error(y_all[te_idx], model.predict(X_all[te_idx])))
        cv_errs.append(np.mean(fold_errs))
    best = degrees[np.argmin(cv_errs)]
    ax.plot(degrees, cv_errs, '-o', lw=2, ms=5, label=f'$k={k}$ (best deg={best})')

ax.set_xlabel('Polynomial degree'); ax.set_ylabel('Mean CV MSE')
ax.set_title('k-Fold CV error vs. degree for different $k$')
ax.legend(fontsize=9)
ax.set_ylim(0, 1.5)

# Right: LOOCV (k=m) vs hold-out for comparison
ax = axes[1]

# Hold-out
split = int(0.7 * m)
idx   = np.random.permutation(m)
X_tr, y_tr = X_all[idx[:split]], y_all[idx[:split]]
X_cv, y_cv = X_all[idx[split:]], y_all[idx[split:]]
holdout_errs = []
for deg in degrees:
    mdl = make_pipeline(PolynomialFeatures(deg), LinearRegression())
    mdl.fit(X_tr, y_tr)
    holdout_errs.append(mean_squared_error(y_cv, mdl.predict(X_cv)))

# LOOCV
loo      = KFold(n_splits=m, shuffle=False)
loo_errs = []
for deg in degrees:
    fold_errs = []
    for tr_idx, te_idx in loo.split(X_all):
        mdl = make_pipeline(PolynomialFeatures(deg), LinearRegression())
        mdl.fit(X_all[tr_idx], y_all[tr_idx])
        fold_errs.append(mean_squared_error(y_all[te_idx], mdl.predict(X_all[te_idx])))
    loo_errs.append(np.mean(fold_errs))

ax.plot(degrees, holdout_errs, 'r-s', lw=2, ms=6,
        label=f'Hold-out (best={degrees[np.argmin(holdout_errs)]})')
ax.plot(degrees, loo_errs,     'b-o', lw=2, ms=6,
        label=f'LOOCV $k=m={m}$ (best={degrees[np.argmin(loo_errs)]})')
ax.set_xlabel('Polynomial degree'); ax.set_ylabel('CV MSE')
ax.set_title('Hold-Out vs. LOOCV\nLOOCV uses more data → more stable estimate')
ax.legend(fontsize=9)
ax.set_ylim(0, 1.5)

fig.suptitle('k-Fold and Leave-One-Out Cross Validation', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. CV as a Diagnostic: Variance vs. Bias

Cross validation curves reveal the nature of the problem:

| Pattern | Diagnosis | Fix |
|---|---|---|
| Train error high, CV error high | High bias (underfitting) | Increase model complexity |
| Train error low, CV error high | High variance (overfitting) | More data, regularise, reduce complexity |
| Both errors low and close | Good generalisation | Ship it |
| CV error has a clear minimum | Model selection works | Pick that degree/hyperparameter |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error

np.random.seed(1)

def true_fn(x):
    return np.sin(2 * np.pi * x)

degrees = list(range(1, 12))
kf      = KFold(n_splits=5, shuffle=True, random_state=0)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
configs   = [
    (10,  0.1, 'Small $m$, low noise\n→ high variance issue'),
    (100, 0.1, 'Large $m$, low noise\n→ good generalisation'),
    (20,  0.8, 'Small $m$, high noise\n→ bias-variance balance'),
]

for ax, (m, noise, title) in zip(axes, configs):
    X = np.sort(np.random.uniform(0, 1, m)).reshape(-1, 1)
    y = true_fn(X.ravel()) + np.random.normal(0, noise, m)

    tr_errs, cv_errs = [], []
    for deg in degrees:
        # Train error (on full data)
        mdl = make_pipeline(PolynomialFeatures(deg), LinearRegression())
        mdl.fit(X, y)
        tr_errs.append(mean_squared_error(y, mdl.predict(X)))

        # 5-fold CV error
        fe = []
        for tri, tei in kf.split(X):
            mdl2 = make_pipeline(PolynomialFeatures(deg), LinearRegression())
            mdl2.fit(X[tri], y[tri])
            fe.append(mean_squared_error(y[tei], mdl2.predict(X[tei])))
        cv_errs.append(np.mean(fe))

    best = degrees[np.argmin(cv_errs)]
    ax.plot(degrees, tr_errs, 'b-o', lw=2, ms=5, label='Train error')
    ax.plot(degrees, cv_errs, 'r-s', lw=2, ms=5, label='5-fold CV error')
    ax.axvline(best, color='green', ls='--', lw=1.5, label=f'Best degree={best}')
    ax.set_title(f'{title}\n($m={m}$, noise={noise})', fontsize=9)
    ax.set_xlabel('Degree'); ax.set_ylabel('MSE')
    ax.legend(fontsize=8.5)
    ax.set_ylim(0, min(3.0, max(cv_errs) * 1.2))

fig.suptitle('5-Fold CV as a Diagnostic Tool', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Derivation Pathway

### Derivation pathway

In [ ]:
from IPython.display import HTML
HTML("""
<svg xmlns="http://www.w3.org/2000/svg" width="780" height="464"
     viewBox="0 0 780 464" font-family="Segoe UI, Arial, sans-serif">
  <defs>
    <marker id="ah" markerWidth="10" markerHeight="7" refX="9" refY="3.5"
            orient="auto" markerUnits="userSpaceOnUse">
      <polygon points="0 0,10 3.5,0 7" fill="#444"/>
    </marker>
    <marker id="ahd" markerWidth="10" markerHeight="7" refX="9" refY="3.5"
            orient="auto" markerUnits="userSpaceOnUse">
      <polygon points="0 0,10 3.5,0 7" fill="#999"/>
    </marker>
  </defs>

  <!-- Row 1: Candidate model set -->
  <rect x="10" y="12" width="185" height="46" rx="7"
        fill="#ffffff" stroke="#2166ac" stroke-width="2"/>
  <text x="102" y="35" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">Candidate models</text>
  <text x="102" y="52" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">M_1, ..., M_d</text>
  <line x1="197" y1="35" x2="216" y2="35"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="12" width="548" height="46" rx="7"
        fill="#eef2f7" stroke="#b0bec5" stroke-width="1.5"/>
  <text x="495" y="40" font-size="13" text-anchor="middle" fill="#333"
        >e.g. polynomial degrees 1&#x2013;10, or different regularisation strengths</text>

  <!-- step 1&#x2192;2 -->
  <line x1="102" y1="58" x2="102" y2="108"
        stroke="#999" stroke-width="1.8" stroke-dasharray="5,3"
        marker-end="url(#ahd)"/>
  <text x="114" y="82" font-size="11.5" font-style="italic" fill="#555"
        >split data into train / held-out partitions</text>

  <!-- Row 2: Hold-out split -->
  <rect x="10" y="112" width="185" height="46" rx="7"
        fill="#ffffff" stroke="#2166ac" stroke-width="2"/>
  <text x="102" y="135" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">Train on S_train</text>
  <text x="102" y="152" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">evaluate on S_cv</text>
  <line x1="197" y1="135" x2="216" y2="135"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="112" width="548" height="46" rx="7"
        fill="#eef2f7" stroke="#b0bec5" stroke-width="1.5"/>
  <text x="495" y="140" font-size="13" text-anchor="middle" fill="#333"
        >hold-out CV: one 70/30 split &#x2014; k-fold CV: k rotations, each fold held out once</text>

  <!-- step 2&#x2192;3 -->
  <line x1="102" y1="158" x2="102" y2="208"
        stroke="#999" stroke-width="1.8" stroke-dasharray="5,3"
        marker-end="url(#ahd)"/>
  <text x="114" y="182" font-size="11.5" font-style="italic" fill="#555"
        >select M_i with lowest CV error</text>

  <!-- Row 3: Model selection -->
  <rect x="10" y="212" width="185" height="46" rx="7"
        fill="#ffffff" stroke="#2166ac" stroke-width="2"/>
  <text x="102" y="240" font-size="13.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">Model selection</text>
  <line x1="197" y1="235" x2="216" y2="235"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="212" width="548" height="46" rx="7"
        fill="#eef2f7" stroke="#b0bec5" stroke-width="1.5"/>
  <text x="495" y="240" font-size="13" text-anchor="middle" fill="#333"
        >argmin_i &#x3b5;&#x302;_S_cv(h_i) &#x2014; CV error is unbiased estimator of generalisation error</text>

  <!-- step 3&#x2192;4 -->
  <line x1="102" y1="258" x2="102" y2="308"
        stroke="#999" stroke-width="1.8" stroke-dasharray="5,3"
        marker-end="url(#ahd)"/>
  <text x="114" y="282" font-size="11.5" font-style="italic" fill="#555"
        >retrain selected model on full S</text>

  <!-- Row 4: Final model -->
  <rect x="10" y="312" width="185" height="46" rx="7"
        fill="#ffffff" stroke="#2166ac" stroke-width="2"/>
  <text x="102" y="340" font-size="13.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">Final model</text>
  <line x1="197" y1="335" x2="216" y2="335"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="312" width="548" height="46" rx="7"
        fill="#eef2f7" stroke="#b0bec5" stroke-width="1.5"/>
  <text x="495" y="340" font-size="13" text-anchor="middle" fill="#333"
        >retrain selected M_i on all of S &#x2014; uses maximum data for deployment</text>

  <!-- step 4&#x2192;5 -->
  <line x1="102" y1="358" x2="102" y2="408"
        stroke="#999" stroke-width="1.8" stroke-dasharray="5,3"
        marker-end="url(#ahd)"/>

  <!-- Row 5: Diagnostics -->
  <rect x="10" y="412" width="185" height="46" rx="7"
        fill="#1a5fa8" stroke="#1a5fa8" stroke-width="2"/>
  <text x="102" y="440" font-size="13.5" font-weight="700"
        text-anchor="middle" fill="#ffffff">CV diagnostics</text>
  <line x1="197" y1="435" x2="216" y2="435"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="412" width="548" height="46" rx="7"
        fill="#dce8f8" stroke="#7aadd4" stroke-width="1.5"/>
  <text x="495" y="440" font-size="13" text-anchor="middle" fill="#333"
        >high bias: both errors high; high variance: CV &#x226b; train; good: both errors low</text>
</svg>
""")

## Summary

| Method | Split | Train count | Use when |
|---|---|---|---|
| Hold-out CV | 70% train / 30% CV (once) | $d$ (one per model) | Large dataset |
| $k$-fold CV | $k$ rotations of $1/k$ held-out fold | $d \times k$ | Small/medium dataset |
| LOOCV ($k=m$) | Each example held out once | $d \times m$ | Very small dataset |

| Concept | Description | Key Insight |
|---|---|---|
| Why train error fails | Always decreases with complexity | Never use train error for model selection |
| CV error | Held-out error averaged over folds | Unbiased estimator of generalisation error |
| Bias diagnosis | Both train and CV error high | Increase model complexity |
| Variance diagnosis | CV error $\gg$ train error | Regularise, get more data, or reduce complexity |
| Final training | Retrain selected model on all of $S$ | Maximises training data for deployment |

**Key insight:** cross validation replaces the biased training error with an unbiased held-out error estimate; $k$-fold CV with $k \approx 10$ is the standard choice that balances bias and computational cost.